In [41]:
import numpy as np
import time

CPU cycles are the basic unit of work for a processor. The CPU has a clock that ticks at a fixed rate, and each tick is one cycle. A 3 GHz CPU ticks 3 billion times per second. An instruction takes one or more cycles to complete, depending on its complexity and things like cache misses, so a simple add might take one cycle while a load from main memory can stall for hundreds.

CPU time is the actual time a CPU spends executing your program's instructions. It doesn't count time the process spent waiting on disk, network, or sleeping, or waiting for its turn on the CPU. It's usually split into two parts:

User time: running your own code
System time: running kernel code on your behalf (system calls, file I/O handling, and so on)

The two are linked by a standard formula:

CPU time = (instruction count × cycles per instruction) / clock rate
         = total cycles / clock rate

So a program that needs 6 billion cycles on a 3 GHz CPU uses 2 seconds of CPU time. You can reduce it by cutting instructions (better algorithm or compiler), lowering cycles per instruction (better cache use, fewer branch mispredictions), or raising the clock rate.

CPU time is different from wall-clock (elapsed) time, which is what a stopwatch measures. A program that spends most of its time waiting on a database might run 10 seconds of wall time but only 0.5 seconds of CPU time. With multithreading it can go the other way: 4 threads busy for 5 seconds each give 20 seconds of CPU time in about 5 seconds of wall time.

In [42]:
def performance(func):
    def wrapper(*args, **kwargs):
        cpu_time = time.process_time()
        wall_time = time.perf_counter()
        result = func(*args, **kwargs)
        cpu_time = time.process_time() - cpu_time
        wall_time = time.perf_counter() - wall_time
        print(f"CPU time: {cpu_time} seconds")
        print(f"Wall time: {wall_time} seconds")
        #return result
    return wrapper

In [43]:
np_arr = np.arange(1_000_000)

list_arr = list(range(1_000_000))


@performance
def multiply_elements(arr):
    return arr * 2

@performance
def multiply_elements_list(arr):
    return [x * 2 for x in arr]







In [44]:
multiply_elements(np_arr)


CPU time: 0.0013690000000003977 seconds
Wall time: 0.0007949160062707961 seconds


In [45]:
multiply_elements_list(list_arr)

CPU time: 0.0183649999999993 seconds
Wall time: 0.01998283399734646 seconds


In [46]:
my_arr = np.arange(1_000_000)

my_list = list(range(1_000_000))

%timeit my_arr2 = my_arr * 2

287 μs ± 3.31 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [47]:
%timeit my_list2 = [x * 2 for x in my_list]

13.5 ms ± 123 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [48]:
my_list = [1, 2, 3, 4, 5]

arr1 = np.array(my_list)

sample_txt = "1 2 3 4 5"

arr2 = np.array(sample_txt)

arr1, arr2

(array([1, 2, 3, 4, 5]), array('1 2 3 4 5', dtype='<U9'))

For arr2 This code will create a 0-dimensional (scalar) NumPy array containing the entire string as a single element.

what is dtype='<U9' in numpy result


Deconstructed numpy dtype notation into byte order, string type, and character limit
Deconstructed numpy dtype notation into byte order, string type, and character limit
It's NumPy's way of describing a fixed-width Unicode string type. Read it in three parts:

< — byte order: little-endian (for multi-byte types). NumPy stores each character as a 4-byte code point, so endianness applies.
U — the kind code: Unicode string.
9 — the itemsize in characters, not bytes. Each element holds up to 9 characters (36 bytes on disk).

In [49]:
arr1.shape, arr2.shape

((5,), ())

In [50]:
data2 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

arr2 = np.array(data2)

arr2.shape, arr2.ndim, arr2.size, arr2.dtype



((3, 3), 2, 9, dtype('int64'))

In [51]:
arr2[:]





array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [52]:
arr3 = np.array([3.14, 4.15, 5.16])

arr3.dtype

dtype('float64')

In [53]:
arr3 = np.array([3.14, 4.15, 5.16])
arr3.dtype = np.int32
arr3.dtype
print(arr3)


arr3 = np.array([3.14, 4.15, 5.16])
arr4 = arr3.astype(np.int32)
arr4.dtype
print(arr4)







[ 1374389535  1074339512 -1717986918  1074829721   171798692  1075094487]
[3 4 5]


Here is exactly what happens under the hood in both scenarios.

1. arr3.dtype = np.int32 (Memory Reinterpretation)
When you directly assign a new value to .dtype, you are doing something called view casting.

You are telling NumPy: "Do not change the 1s and 0s in memory. Just put on different glasses and read those exact same bytes as if they were 32-bit integers."

Python
arr3 = np.array([3.14, 4.15, 5.16]) # Defaults to float64 (8 bytes per number)
arr3.dtype = np.int32               # Reinterpret as int32 (4 bytes per number)
Because a default float is 64 bits (8 bytes) and you are telling it to read in 32-bit (4 byte) chunks, two things go wrong:

Garbage Values: The binary pattern for the floating-point number 3.14 (IEEE 754 standard) looks completely different from the binary pattern for the integer 3. When you force the CPU to read float bytes as integer bytes, you get massive, seemingly random numbers (like 1374389535).

Array Size Doubles: Because you had three 8-byte chunks (24 bytes total), and you are now reading them in 4-byte chunks, your array will suddenly have 6 elements instead of 3.

2. arr3.astype(np.int32) (Value Conversion)
This is the standard, safe way to change data types.


When you use .astype(), you are doing data conversion.

You are telling NumPy: "Read the actual mathematical value of this float, round it down to an integer, create a brand new block of memory, and write the correct integer binary pattern into it."

NumPy reads 3.14, calculates the integer equivalent 3, allocates new RAM, and writes the standard binary for 3.

In [54]:
numeric_strings = np.array(['1.23', '4.56', '7.8956'], dtype=np.bytes_)
numeric_strings.dtype



dtype('S6')

In [55]:
numeric = numeric_strings.astype(np.float64)
numeric

array([1.23  , 4.56  , 7.8956])

In [56]:
l1 = [1, 2, 3, 4, 5,6,7,8]

arr1 = np.array([1, 2, 3, 4, 5,6,7,8])

l1[1:4] = [122]

arr1[1:4] = 122

In [57]:
l1,arr1

([1, 122, 5, 6, 7, 8], array([  1, 122, 122, 122,   5,   6,   7,   8]))

Notice in the numpy array slicing the value is broadcasted

In [58]:
arr2 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

arr2[0]

array([1, 2, 3])

In [59]:
arr2[0].shape,arr2[0].ndim

((3,), 1)

In [60]:
arr2[0:2]

array([[1, 2, 3],
       [4, 5, 6]])

In [61]:
arr2[0][2], arr2[0,2] #Same thing

(np.int64(3), np.int64(3))

In [62]:
arr3d = np.array([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])

arr3d

array([[[ 1,  2,  3],
        [ 4,  5,  6]],

       [[ 7,  8,  9],
        [10, 11, 12]]])

In [63]:
arr3d.shape

(2, 2, 3)

In [64]:
arr3d[0]

array([[1, 2, 3],
       [4, 5, 6]])

In [65]:
old_values = arr3d[0].copy()
arr3d[0] = 42

print(arr3d)

print(old_values)

[[[42 42 42]
  [42 42 42]]

 [[ 7  8  9]
  [10 11 12]]]
[[1 2 3]
 [4 5 6]]


In [66]:
arr3d[0] = old_values

arr3d

array([[[ 1,  2,  3],
        [ 4,  5,  6]],

       [[ 7,  8,  9],
        [10, 11, 12]]])

In [67]:
arr3d[0,1], arr3d[0,1,2]

(array([4, 5, 6]), np.int64(6))

In [68]:
arr2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

arr2d[2,:], arr2d[2:,:]


(array([7, 8, 9]), array([[7, 8, 9]]))

In [69]:
arr2d[2,:].shape,arr2d[2:,:].shape

((3,), (1, 3))

In [70]:
arr2d[2,:].ndim,arr2d[2:,:].ndim

(1, 2)

Notice the difference between `arr2d[2,:]` and `arr2d[2:,:]`: the values are the same, but the latter gives a 2D array.


In [71]:
arr_new = arr2d[2,:]

arr_new

array([7, 8, 9])

In [72]:
arr_new[1] = 10

arr_new, arr2d

(array([ 7, 10,  9]),
 array([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7, 10,  9]]))

#### Views vs copies


In [73]:
arr_b = arr_new[np.array([True, False, True])   ]

arr_b 

array([7, 9])

In [74]:
arr_b[1] = 100 # Does not change original array

arr_b,arr2d,arr_new

(array([  7, 100]),
 array([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7, 10,  9]]),
 array([ 7, 10,  9]))

In [75]:
a = np.zeros((8,4))

for i in range(8):
    a[i] = i
a



array([[0., 0., 0., 0.],
       [1., 1., 1., 1.],
       [2., 2., 2., 2.],
       [3., 3., 3., 3.],
       [4., 4., 4., 4.],
       [5., 5., 5., 5.],
       [6., 6., 6., 6.],
       [7., 7., 7., 7.]])

In [76]:
print(a[[4,0]] )#Fancy indexing
a[[4,0]] = 100

print(a)

a[[4,0]] = a[[1,2]]

print(a)



[[4. 4. 4. 4.]
 [0. 0. 0. 0.]]
[[100. 100. 100. 100.]
 [  1.   1.   1.   1.]
 [  2.   2.   2.   2.]
 [  3.   3.   3.   3.]
 [100. 100. 100. 100.]
 [  5.   5.   5.   5.]
 [  6.   6.   6.   6.]
 [  7.   7.   7.   7.]]
[[2. 2. 2. 2.]
 [1. 1. 1. 1.]
 [2. 2. 2. 2.]
 [3. 3. 3. 3.]
 [1. 1. 1. 1.]
 [5. 5. 5. 5.]
 [6. 6. 6. 6.]
 [7. 7. 7. 7.]]


In [77]:
arr = np.arange(32).reshape((8,4))

arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23],
       [24, 25, 26, 27],
       [28, 29, 30, 31]])

In [78]:
arr[[1,5,7,2],[0,3,1,2]] # arr[1,0] = 4  arr[5,3] = 23.. 

array([ 4, 23, 29, 10])

In [79]:
arr[[1,5,7,2]]

array([[ 4,  5,  6,  7],
       [20, 21, 22, 23],
       [28, 29, 30, 31],
       [ 8,  9, 10, 11]])

In [80]:
arr[[1,5,7,2]][:,[0,3]]

array([[ 4,  7],
       [20, 23],
       [28, 31],
       [ 8, 11]])